## Parte 4 · HTML (20 %)
Con el fichero local portal_transparencia.html de los recursos.

1. Extrae la tabla de contratos con BeautifulSoup.

2. Señala qué etiquetas son semánticas y cuáles son solo de presentación, y explica por qué la diferencia importa para poder extraer datos y para la accesibilidad.

3. Convierte la tabla a la misma estructura que las partes anteriores.

### 4.1. Añadir la ruta del HTML a "analisis.yaml"



Para no dejar rutas a mano, se amplía el YAML de la parte 3 con una nueva entrada:

```yaml
entradas:
  json: "datos/licitaciones_api.json"
  xml:  "datos/licitaciones_pcsp.xml"
  html: "datos/portal_transparencia.html"

### 4.2. Extracción con BeautifulSoup

In [ ]:

from bs4 import BeautifulSoup
from datetime import datetime

with open(RUTA_HTML, encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

tabla = soup.find("table", id="tabla-contratos")
filas = tabla.select("tbody tr")

contratos = []
for tr in filas:
    celdas = tr.find_all("td")

    id_       = tr.get("data-expediente") or celdas[0].get_text(strip=True)
    organo    = celdas[1].get_text(strip=True)
    objeto    = celdas[2].get_text(strip=True)
    importe   = float(celdas[3]["data-valor"])         # dato limpio del atributo
    fecha_iso = celdas[4].find("time")["datetime"]     # dato limpio del atributo
    fecha     = datetime.fromisoformat(fecha_iso)

    contratos.append({
        "id": id_,
        "organo": organo,
        "objeto": objeto,
        "importe": importe,
        "fechaLimite": fecha,
    })

for c in contratos:
    print(c)

### 4.3. Etiquetas semánticas vs. de presentación

| Tipo | Etiquetas / atributos | Ejemplo en el documento | Para qué sirve |
|---|---|---|---|
| **Semántica** | `<main>`, `<section>` | `<section id="contratos">` | Estructura del contenido |
| **Semántica** | `<h1>`, `<h2>`, `<p>` | `<h1>Contratos menores…</h1>` | Jerarquía y texto |
| **Semántica** | `<table>`, `<thead>`, `<tbody>`, `<tr>`, `<th>`, `<td>`, `<caption>` | `<table id="tabla-contratos">` | Estructura tabular |
| **Semántica** | `<time datetime="...">` | `<time datetime="2026-10-18">18/10/2026</time>` | Fecha legible + máquina |
| **Semántica** | `scope="col"` | `<th scope="col">Expediente</th>` | Cabecera de columna |
| **Semántica** | `data-expediente`, `data-valor` | `<td data-valor="14800.00">` | Dato limpio para máquinas |
| **Semántica** | `<ul>`, `<li>`, `<small>` | `<small>Los importes…</small>` | Lista y texto secundario |
| **Presentación** | `<b>`, `<i>` | `<b>Portal de transparencia municipal</b>` | Solo negrita/cursiva |
| **Presentación** | `<font size="2">` | `<font size="2">Ultima actualizacion…</font>` | Obsoleto, solo tamaño |
| **Presentación** | clases visuales | `class="titulo-grande"`, `"subtitulo"`, `"cabecera-portal"`, `"pie-portal"`, `"nota-pie"`, `"importe"` | Solo estilo |

| Aspecto | Por qué importa |
|---|---|
| **Extracción** | Las semánticas dan un contrato estable (`table#tabla-contratos tbody tr`, `time[datetime]`, `data-valor`); las de presentación cambian con el diseño y no dicen qué contiene el nodo. |
| **Datos limpios** | El texto visible es para humanos (`"14.800,00 EUR"`, `"18/10/2026"`); los atributos `data-valor` y `datetime` son para máquinas y evitan parsear formato local. |
| **Accesibilidad** | El lector de pantalla usa `<h1>`, `<th scope="col">`, `<caption>` y `<time datetime>`; `<b>`, `<i>` y `<font>` no aportan nada al usuario. |